In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import midi_sonify as ms

In [ ]:
pm = ms.load_midi("../data/hymns/Amazing_Grace_(nc)pope.mid")
print(f"Instruments: {len(pm.instruments)}")
print(f"Duration:    {pm.get_end_time():.1f} s")
print(f"Tempo:       {pm.estimate_tempo():.0f} BPM")

In [ ]:
instruments = ms.list_instruments(pm)

# Print a formatted table
header = f"{'Idx':>3}  {'Name':<16} {'Prog':>4}  {'Notes':>5}  {'Pitch':>9}  {'Time':>13}  {'Poly':>5}"
print(header)
print("-" * len(header))
for info in instruments:
    pmin = info['pitch_min']
    pmax = info['pitch_max']
    pitch_str = f"{pmin:>3}–{pmax:<3}" if pmin is not None else "   –   "
    tstart = info['start_time']
    tend = info['end_time']
    time_str = f"{tstart:5.1f}–{tend:5.1f}" if tstart is not None else "     –     "
    print(
        f"{info['instrument_index']:>3}  {info['name']:<16} {info['program']:>4}  "
        f"{info['note_count']:>5}  {pitch_str}  {time_str}  "
        f"{info['approximate_polyphony']:>5.1f}"
    )

In [ ]:
# Select by substring
violin_pm = ms.select_instruments(pm, "violin")
print("Substring select 'violin':")
for info in ms.list_instruments(violin_pm):
    print(f"  [{info['instrument_index']}] {info['name']} — {info['note_count']} notes")

# Select by index
first_pm = ms.select_instruments(pm, 0)
print("\nIndex select [0]:")
for info in ms.list_instruments(first_pm):
    print(f"  [{info['instrument_index']}] {info['name']} — {info['note_count']} notes")

In [ ]:
# Full piece — inline playback
audio_full = ms.synthesize(pm)
print(f"Audio shape: {audio_full.shape}, duration: {len(audio_full)/44100:.1f} s")
ms.play_audio(audio_full)

In [ ]:
# Trim to first 10 seconds
pm_10s = ms.trim_time(pm, 0.0, 10.0)
audio_10s = ms.synthesize(pm_10s)
print(f"Trimmed to 10 s — audio length: {len(audio_10s)/44100:.1f} s")
ms.play_audio(audio_10s)

In [ ]:
# Estimate measure times
mtimes = ms.estimate_measure_times(pm)
print(f"Total measures: {len(mtimes)}")
print("First 8 measure start times (s):")
for i, t in enumerate(mtimes[:8], 1):
    print(f"  Measure {i}: {t:.2f} s")

# Trim measures 1–5
pm_m1_5 = ms.trim_measures(pm, 1, 5)
audio_m1_5 = ms.synthesize(pm_m1_5)
print(f"\nMeasures 1–5 — audio length: {len(audio_m1_5)/44100:.1f} s")
ms.play_audio(audio_m1_5)

In [ ]:
# Chained: select "cello" + trim first 15 s
cello_pm = ms.select_instruments(pm, "cello")
cello_15s = ms.trim_time(cello_pm, 0.0, 15.0)
audio_cello = ms.synthesize(cello_15s)
print(f"Cello first 15 s — audio length: {len(audio_cello)/44100:.1f} s")
ms.play_audio(audio_cello)

In [ ]:
# Write WAV
ms.write_wav("../output/amazing_grace_first_10s.wav", audio_10s)
import pathlib
wav_path = pathlib.Path("../output/amazing_grace_first_10s.wav")
print(f"Written: {wav_path}  ({wav_path.stat().st_size / 1024:.0f} KB)")

In [ ]:
# FluidSynth playback — realistic sampled instruments
SF2_PATH = "../data/soundfonts/GeneralUser_GS.sf2"

audio_fluid = ms.synthesize(pm, sf2_path=SF2_PATH)
print(f"FluidSynth audio: {len(audio_fluid)/44100:.1f} s")
ms.play_audio(audio_fluid)

In [ ]:
# Side-by-side comparison: sine waves vs FluidSynth (first 10 seconds)
from IPython.display import display, HTML

pm_10s = ms.trim_time(pm, 0.0, 10.0)

display(HTML("<b>Sine-wave synthesis (robotic, discrete notes):</b>"))
audio_sine = ms.synthesize(pm_10s)
display(ms.play_audio(audio_sine))

display(HTML("<b>FluidSynth + SoundFont (natural, connected):</b>"))
audio_sf = ms.synthesize(pm_10s, sf2_path=SF2_PATH)
display(ms.play_audio(audio_sf))